In [1]:
# ==========================================================================
# Appendix Chatbot : 한국어 GPT 챗봇 서비스 (정리본)
#   모델 : skt/kogpt2-base-v2 (CPU)
#   서버 : FastAPI app/chatbot_api.py -> http://localhost:8000
#   화면 : Streamlit frontend/app_chatbot.py -> http://localhost:8501
#   인증 : X-API-Key: dev-key-001 (app/auth.py, 서빙 프로젝트와 공용)
#   커널 : Python 3.12 (WSL model-serving) = .venv-wsl312
#   흐름 : 모델 확인 -> 파일 3개 생성(모델, 스키마, API) -> 서버 기동 -> API 테스트 -> 화면 -> 통합 테스트
# ==========================================================================
# [1. 환경 확인] 커널이 .venv-wsl312 인지, 작업 폴더가 프로젝트 폴더인지 먼저 봅니다
import os, sys
import torch, transformers

print("파이썬      :", sys.executable)          # .venv-wsl312/bin/python 이어야 합니다
print("작업 폴더   :", os.getcwd())             # model-serving-course 폴더여야 합니다
print("PyTorch     :", torch.__version__)
print("transformers:", transformers.__version__)  # torch 2.4 와 맞는 4.x
print("GPU 사용    :", torch.cuda.is_available())  # False 면 CPU 용 kogpt2 를 씁니다

/mnt/c/EPA1st/model-serving-course/.venv-wsl312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


파이썬      : /mnt/c/EPA1st/model-serving-course/.venv-wsl312/bin/python
작업 폴더   : /mnt/c/EPA1st/model-serving-course
PyTorch     : 2.4.1+cpu
transformers: 4.57.6
GPU 사용    : False


In [2]:
# ==========================================================================
# [2. 노트북에서 모델 직접 써 보기]
#   GPU가 있으면 1.2B 모델, 없으면 kogpt2를 고릅니다. 첫 실행은 약 0.5GB를 받습니다
# ==========================================================================
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

# GPU 환경이면 큰 모델, CPU 환경이면 작은 모델
MODEL_NAME = "skt/ko-gpt-trinity-1.2B-v0.5" if torch.cuda.is_available() else "skt/kogpt2-base-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_NAME)               # 토크나이저
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, use_safetensors=True)    # 모델 (safetensors 형식)
model = model.to(device).eval()                                              # 추론 모드

print(f"모델: {MODEL_NAME} ({device})")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


모델: skt/kogpt2-base-v2 (cpu)
파라미터 수: 125,164,032


In [3]:
# [2.1 한 문장 이어 쓰기] generate() 옵션이 결과를 좌우합니다. 반복 억제 옵션 두 개를 넣었습니다
prompt = "인공지능의 미래는"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        attention_mask=torch.ones_like(input_ids),  # 경고를 없앱니다 (패딩이 없으므로 전부 1)
        max_new_tokens=50,          # 새로 만들 최대 토큰 수
        do_sample=True,             # 확률적으로 고릅니다 (False 면 매번 같은 답)
        temperature=0.8,            # 낮을수록 무난, 높을수록 엉뚱
        top_k=50, top_p=0.9,        # 후보를 상위 k개 / 누적확률 p 안으로 제한
        repetition_penalty=1.3,     # 이미 나온 토큰의 확률을 낮춤
        no_repeat_ngram_size=3,     # 같은 3토큰 묶음 반복 금지
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


인공지능의 미래는 이미 시작되었다고 해도 과언이 아닐 정도로 지능이 발달한 세상이다. 이브 생가에는 그 동안 알려지지 않은 희귀 동물들이 많이 전시되어 있다.
그중 한 명인 아쿠아마린과는 서로 다른 성격의 동물로 태어난다.
하지만 아쿠아가이아


In [4]:
# [2.2 대화 기록을 프롬프트 하나로 합치기]
#   모델은 대화를 모릅니다. '사용자:/봇:' 형식의 글로 만들어 이어 쓰게 합니다
conversation = [
    "사용자: 안녕하세요!",
    "봇: 안녕하세요! 무엇을 도와드릴까요?",
    "사용자: 오늘 날씨가 어때?",
]
prompt = "\n".join(conversation) + "\n봇:"     # 마지막을 "봇:" 으로 끝내 봇 차례를 만듭니다

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    output_ids = model.generate(
        input_ids, attention_mask=torch.ones_like(input_ids),
        max_new_tokens=50, do_sample=True, temperature=0.8, top_k=50,
        repetition_penalty=1.3, no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
    )

# 새로 만든 토큰만 꺼내고, 첫 줄만 봇의 답으로 씁니다
new_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
lines = [l.strip() for l in new_text.split("\n") if l.strip()]
print("봇 응답:", lines[0] if lines else "(없음)")

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


봇 응답: 지금 뭐라고 하셨지요?


In [5]:
%%writefile app/chatbot_model.py
# [3. 서버 파일 만들기]
# [3.1 모델 클래스 app/chatbot_model.py (1/5)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   노트북에서 해 본 일을 클래스로 묶습니다. 서버는 이 클래스만 압니다
#   모델을 바꿀 때 고칠 곳은 여기 한 곳입니다
#   ChatbotModel : Hugging Face 한국어 GPT 모델을 로드하고 텍스트를 생성합니다
"""
Day 7 - 한국어 GPT 챗봇 모델
"""
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel


class ChatbotModel:
    """Hugging Face 한국어 GPT 모델을 로드하고 텍스트를 생성합니다."""

Overwriting app/chatbot_model.py


In [6]:
%%writefile -a app/chatbot_model.py

# [app/chatbot_model.py (2/5)] __init__() : 장치 선택, 토크나이저/모델 로드
    def __init__(self, model_name: str = "skt/kogpt2-base-v2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
        self.model = GPT2LMHeadModel.from_pretrained(model_name, use_safetensors=True)
        self.model = self.model.to(self.device)
        self.model.eval()

        self.model_name = model_name

Appending to app/chatbot_model.py


In [7]:
%%writefile -a app/chatbot_model.py

# [app/chatbot_model.py (3/5)] generate_response() : 대화 기록을 받아 응답을 생성합니다
#   대화 기록 -> 프롬프트 구성 -> 토크나이징 -> 토큰 수 제한(모델 최대 길이 초과 방지)
    def generate_response(
        self,
        messages: list[dict],
        max_new_tokens: int = 100,
        temperature: float = 0.8,
        top_k: int = 50,
        top_p: float = 0.9,
    ) -> str:
        """
        대화 기록을 받아 응답을 생성합니다.

        Args:
            messages: [{"role": "user", "content": "안녕"}, {"role": "bot", "content": "안녕하세요!"}, ...]
            max_new_tokens: 최대 생성 토큰 수
            temperature: 생성 다양성
        Returns:
            생성된 응답 텍스트
        """
        # 대화 기록 → 프롬프트 구성
        prompt = self._build_prompt(messages)              # 프롬프트 구성

        # 토크나이징
        input_ids = self.tokenizer.encode(
            prompt, return_tensors="pt"
        ).to(self.device)

        # 토큰 수 제한 (모델 최대 길이 초과 방지)
        max_length = getattr(self.model.config, "n_positions", 1024)

Appending to app/chatbot_model.py


In [8]:
%%writefile -a app/chatbot_model.py

# [app/chatbot_model.py (4/5)] generate_response() 이어서
#   최근 대화만 유지 -> 생성 -> 생성된 부분만 디코딩
#   -> "사용자:" 이후는 자름, 봇의 한 턴은 한 줄로 봄 (다음 줄부터는 모델이 이어 쓴 다른 글)
        if input_ids.shape[1] > max_length - max_new_tokens:
            # 최근 대화만 유지
            input_ids = input_ids[:, -(max_length - max_new_tokens):]

        # 텍스트 생성
        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,             # 생성 파라미터
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                do_sample=True,
                attention_mask=torch.ones_like(input_ids),
                pad_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.3,     # 이미 나온 토큰이 다시 나올 확률을 낮춤
                no_repeat_ngram_size=3,     # 같은 3토큰 묶음이 두 번 나오지 않게 막음
            )

        # 디코딩: 생성된 부분만 추출
        new_ids = output_ids[0][input_ids.shape[1]:]
        response = self.tokenizer.decode(new_ids, skip_special_tokens=True).strip()

        # "사용자:" 이후 텍스트가 나오면 거기까지만 자름
        if "사용자:" in response:
            response = response.split("사용자:")[0].strip()

        # 봇의 한 턴은 한 줄로 봅니다. 다음 줄부터는 모델이 이어 쓴 다른 글입니다
        lines = [l.strip() for l in response.split("\n") if l.strip()]

Appending to app/chatbot_model.py


In [9]:
%%writefile -a app/chatbot_model.py

# [app/chatbot_model.py (5/5)] generate_response() 마무리
#   _build_prompt() : 대화 기록을 프롬프트 문자열로 변환합니다
        response = lines[0] if lines else ""

        return response if response else "(응답을 생성하지 못했습니다)"

    def _build_prompt(self, messages: list[dict]) -> str:
        """대화 기록을 프롬프트 문자열로 변환합니다."""
        lines = []
        for msg in messages:
            role = "사용자" if msg["role"] == "user" else "봇"
            lines.append(f"{role}: {msg['content']}")
        lines.append("봇:")  # 모델이 이어서 생성하도록
        return "\n".join(lines)

Appending to app/chatbot_model.py


In [10]:
%%writefile app/chatbot_schemas.py
# [3.2 요청/응답 규격 app/chatbot_schemas.py (1/3)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   Pydantic이 입력을 검증합니다. 범위를 벗어나면 FastAPI가 자동으로 422 를 돌려줍니다
#   Message : 단일 대화 메시지
"""
Day 7 - 챗봇 API 스키마
"""
from pydantic import BaseModel, Field
from typing import Optional


class Message(BaseModel):
    """단일 대화 메시지"""
    role: str = Field(..., description="역할: 'user' 또는 'bot'")
    content: str = Field(..., min_length=1, description="메시지 내용")

Overwriting app/chatbot_schemas.py


In [11]:
%%writefile -a app/chatbot_schemas.py

# [app/chatbot_schemas.py (2/3)] ChatRequest : 챗봇 요청 (대화 기록, 생성 옵션의 범위 제한)
class ChatRequest(BaseModel):
    """챗봇 요청"""
    messages: list[Message] = Field(
        ...,
        min_length=1,
        description="대화 기록. 마지막 메시지가 사용자의 현재 입력.",
    )
    max_new_tokens: int = Field(default=100, ge=10, le=500)     # 범위 제한
    temperature: float = Field(default=0.8, gt=0.0, le=2.0)

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "messages": [
                        {"role": "user", "content": "안녕하세요!"}
                    ],
                    "max_new_tokens": 100,
                    "temperature": 0.8,
                }
            ]
        }
    }

Appending to app/chatbot_schemas.py


In [12]:
%%writefile -a app/chatbot_schemas.py

# [app/chatbot_schemas.py (3/3)] ChatResponse : 챗봇 응답
#   user 는 app/auth.py 가 돌려주는 {사용자, 부서} dict 입니다
class ChatResponse(BaseModel):
    """챗봇 응답"""
    success: bool = Field(description="성공 여부")
    response: str = Field(description="생성된 응답 텍스트")
    model_name: str = Field(description="사용된 모델")
    user: Optional[dict] = Field(default=None, description="인증된 사용자 정보 (사용자, 부서)")

Appending to app/chatbot_schemas.py


In [13]:
%%writefile app/chatbot_api.py
# [3.3 API 서버 app/chatbot_api.py (1/5)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   - 시작할 때 모델을 한 번만 올립니다 (startup)
#   - /chat 은 Depends(verify_api_key) 로 인증합니다 (키가 없거나 틀리면 401)
#   - 생성은 무거운 동기 작업이라 run_in_executor 로 별도 스레드에서 돌려 서버가 멈추지 않게 합니다
#   이 셀 : import 와 앱/스레드풀 설정
"""
Day 7 - 한국어 챗봇 FastAPI 서버
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException

from app.chatbot_schemas import ChatRequest, ChatResponse
from app.chatbot_model import ChatbotModel
from app.auth import verify_api_key
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


# ===== 설정 =====
logger = setup_logger("chatbot_api")

app = FastAPI(
    title="Korean Chatbot API",
    description="한국어 GPT 기반 멀티턴 챗봇 API (인증 필요)",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="chatbot")

Overwriting app/chatbot_api.py


In [14]:
%%writefile -a app/chatbot_api.py

# [app/chatbot_api.py (2/5)] startup() : 서버 시작 시 1회 실행, 모델 로드
# ===== 모델 로드 =====
chatbot = None

@app.on_event("startup")
async def startup():
    global chatbot
    import torch
    model_name = "skt/ko-gpt-trinity-1.2B-v0.5" if torch.cuda.is_available() else "skt/kogpt2-base-v2"
    logger.info(f"챗봇 모델 로드 중: {model_name}")
    chatbot = ChatbotModel(model_name)                # ChatbotModel 인스턴스 생성
    logger.info("모델 로드 완료")

Appending to app/chatbot_api.py


In [15]:
%%writefile -a app/chatbot_api.py

# [app/chatbot_api.py (3/5)] run_chat() : 별도 스레드에서 실행되는 추론 함수
def run_chat(messages, max_new_tokens, temperature):
    """별도 스레드에서 실행되는 추론 함수"""
    if chatbot is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
    return chatbot.generate_response(
        messages=[m.model_dump() for m in messages],
        max_new_tokens=max_new_tokens,
        temperature=temperature,
    )


# ===== 엔드포인트 =====

Appending to app/chatbot_api.py


In [16]:
%%writefile -a app/chatbot_api.py

# [app/chatbot_api.py (4/5)] GET /health : 모델이 올라왔으면 healthy, 아니면 loading
@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if chatbot else "loading",
        "model": chatbot.model_name if chatbot else None,
    }

Appending to app/chatbot_api.py


In [17]:
%%writefile -a app/chatbot_api.py

# [app/chatbot_api.py (5/5)] POST /chat : 대화 기록을 받아 응답을 생성합니다 (인증 적용)
@app.post("/chat", response_model=ChatResponse, tags=["Chat"])
async def chat(
    request: ChatRequest,
    user: dict = Depends(verify_api_key),              # 인증 적용
):
    """대화 기록을 받아 응답을 생성합니다."""
    if chatbot is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    logger.info(f"채팅 요청 — 사용자: {user['사용자']}, 메시지 수: {len(request.messages)}")

    try:
        loop = asyncio.get_event_loop()
        response_text = await loop.run_in_executor(    # 비동기 추론
            inference_executor,
            run_chat,
            request.messages,
            request.max_new_tokens,
            request.temperature,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"텍스트 생성 실패: {str(e)}")

    return ChatResponse(
        success=True,
        response=response_text,
        model_name=chatbot.model_name,
        user=user,
    )

Appending to app/chatbot_api.py


In [18]:
# ==========================================================================
# [4. 서버 띄우기 (1/2)] 도우미 함수
#   port_in_use()     : 포트가 사용 중인지 확인
#   chatbot_healthy() : 8000번에 챗봇 서버가 정상으로 떠 있는지 확인
# ==========================================================================
import threading, time, socket
import requests
import nest_asyncio, uvicorn
nest_asyncio.apply()          # 주피터의 이벤트 루프 안에서 uvicorn 을 돌릴 수 있게 합니다

API_BASE = "http://localhost:8000"
VALID_KEY = "dev-key-001"     # app/auth.py 의 실습용 키 (김개발, 부서: 공개/개발)


def port_in_use(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0


def chatbot_healthy():
    try:
        return requests.get(f"{API_BASE}/health", timeout=2).json().get("status") == "healthy"
    except Exception:
        return False

In [19]:
# [4. 서버 띄우기 (2/2)]
#   8000번이 이미 챗봇 서버로 떠 있으면 그대로 씁니다. 다른 프로그램이 쓰고 있으면 알려 주고 멈춥니다
#   (원본의 time.sleep(10) 대신 /health 가 healthy 가 될 때까지 기다립니다)
if chatbot_healthy():
    print("이미 떠 있는 챗봇 서버를 씁니다:", API_BASE)
elif port_in_use(8000):
    print("8000번을 다른 프로그램이 쓰고 있습니다. 터미널에서 확인: ss -ltnp | grep :8000")
else:
    server_thread = threading.Thread(
        target=lambda: uvicorn.run("app.chatbot_api:app", host="127.0.0.1", port=8000),
        daemon=True)          # 커널이 꺼지면 서버도 같이 꺼집니다
    server_thread.start()
    for _ in range(180):      # 모델 로드가 끝날 때까지 최대 3분 기다립니다
        if chatbot_healthy():
            break
        time.sleep(1)
    print("서버 시작됨:", API_BASE, "| 상태:", "healthy" if chatbot_healthy() else "아직 로딩 중")

이미 떠 있는 챗봇 서버를 씁니다: http://localhost:8000


In [20]:
# ==========================================================================
# [5. API 테스트]
# [5.1 헬스체크와 한 번 대화]
# ==========================================================================
print("헬스체크:", requests.get(f"{API_BASE}/health").json())

resp = requests.post(
    f"{API_BASE}/chat",
    json={"messages": [{"role": "user", "content": "안녕하세요!"}]},
    headers={"X-API-Key": VALID_KEY},     # 인증 헤더
    timeout=120,
)
result = resp.json()
print("상태:", resp.status_code)
print("사용자:", result["user"])          # {'사용자': '김개발', '부서': [...]}
print("응답:", result["response"])

헬스체크: {'status': 'healthy', 'model': 'skt/kogpt2-base-v2'}


상태: 200
사용자: {'사용자': '김개발', '부서': ['공개', '개발']}
응답: 닌자 어드벤처 시리즈


In [21]:
# [5.2 여러 번 주고받기 (멀티턴)]
#   서버는 대화를 기억하지 않습니다(stateless). 클라이언트가 지난 대화를 매번 같이 보냅니다
resp = requests.post(
    f"{API_BASE}/chat",
    json={"messages": [
        {"role": "user", "content": "안녕하세요!"},
        {"role": "bot", "content": result["response"]},      # 이전 응답을 포함
        {"role": "user", "content": "오늘 뭐 하면 좋을까?"},   # 새 질문
    ]},
    headers={"X-API-Key": VALID_KEY},
    timeout=120,
)
print("멀티턴 응답:", resp.json()["response"])

멀티턴 응답: 카오루마루 ( 플레이어 ) , 미츠히메 유우키 ( 슈팅게임 및 아케이드 게임 한정. )


In [22]:
%%writefile frontend/app_chatbot.py
# [6. 화면 frontend/app_chatbot.py (1/5)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   - st.session_state 에 대화 기록을 저장합니다. Streamlit 은 입력할 때마다 스크립트 전체를
#     다시 실행하므로, 여기에 넣지 않으면 기록이 사라집니다
#   - st.chat_message / st.chat_input 으로 채팅 화면을 만듭니다
#   이 셀 : 페이지 설정
"""
Day 7 - 한국어 GPT 챗봇 대시보드
"""
import streamlit as st
import requests


# ===== 페이지 설정 =====
st.set_page_config(
    page_title="한국어 챗봇",
    page_icon="💬",
    layout="centered",
)

API_BASE = "http://localhost:8000"

Overwriting frontend/app_chatbot.py


In [23]:
%%writefile -a frontend/app_chatbot.py

# [frontend/app_chatbot.py (2/5)] call_chat_api() : 챗봇 API를 호출합니다
# ===== API 호출 =====
def call_chat_api(messages, api_key, max_new_tokens=100, temperature=0.8):
    """챗봇 API를 호출합니다."""
    try:
        resp = requests.post(
            f"{API_BASE}/chat",
            json={
                "messages": messages,
                "max_new_tokens": max_new_tokens,
                "temperature": temperature,
            },
            headers={"X-API-Key": api_key},            # 인증 헤더
            timeout=60,   # 텍스트 생성은 시간이 걸릴 수 있음
        )
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.**")
        return None
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            st.error("🔑 **인증 실패.** API Key를 확인하세요.")
        else:
            st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None

Appending to frontend/app_chatbot.py


In [24]:
%%writefile -a frontend/app_chatbot.py

# [frontend/app_chatbot.py (3/5)] 사이드바 : API 키, 생성 옵션, 서버 상태
# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")

    api_key = st.text_input("API Key", value="dev-key-001", type="password")

    st.divider()

    max_tokens = st.slider("최대 생성 토큰", 10, 300, 100, step=10)
    temperature = st.slider("Temperature", 0.1, 2.0, 0.8, step=0.1)

    st.divider()

    # 서버 상태
    try:
        health = requests.get(f"{API_BASE}/health", timeout=3).json()
        if health.get("status") == "healthy":
            st.success(f"🟢 서버 연결됨")
            st.caption(f"모델: {health.get('model', 'N/A')}")
        else:
            st.warning("🟡 모델 로딩 중...")
    except Exception:
        st.error("🔴 서버 연결 실패")

    st.divider()

    if st.button("대화 초기화"):
        st.session_state["chat_messages"] = []         # 대화 기록 초기화
        st.rerun()

    st.caption("Korean GPT Chatbot v1.0")

Appending to frontend/app_chatbot.py


In [25]:
%%writefile -a frontend/app_chatbot.py

# [frontend/app_chatbot.py (4/5)] 대화 기록 초기화 -> 메인 영역 -> 기존 대화 표시 -> 사용자 입력
# ===== 대화 기록 초기화 =====
if "chat_messages" not in st.session_state:
    st.session_state["chat_messages"] = []


# ===== 메인 영역 =====
st.title("💬 한국어 챗봇")
st.write("한국어 GPT 모델과 대화해 보세요.")

# 기존 대화 기록 표시
for msg in st.session_state["chat_messages"]:
    with st.chat_message(msg["role"]):                 # 채팅 메시지 표시
        st.write(msg["content"])

# 사용자 입력
user_input = st.chat_input("메시지를 입력하세요...")     # 채팅 입력란

Appending to frontend/app_chatbot.py


In [26]:
%%writefile -a frontend/app_chatbot.py

# [frontend/app_chatbot.py (5/5)] 사용자 메시지 추가/표시 -> API 형식으로 변환해 호출 -> 봇 응답을 기록에 추가
if user_input:
    # 사용자 메시지 추가 및 표시
    st.session_state["chat_messages"].append({
        "role": "user",
        "content": user_input,
    })
    with st.chat_message("user"):
        st.write(user_input)

    # API 호출
    with st.chat_message("assistant"):
        with st.spinner("생성 중..."):
            # 대화 기록을 API 형식으로 변환
            api_messages = []
            for msg in st.session_state["chat_messages"]:
                role = "user" if msg["role"] == "user" else "bot"
                api_messages.append({"role": role, "content": msg["content"]})

            result = call_chat_api(
                messages=api_messages,
                api_key=api_key,
                max_new_tokens=max_tokens,
                temperature=temperature,
            )

        if result and result.get("success"):
            bot_response = result["response"]
            st.write(bot_response)

            # 봇 응답을 대화 기록에 추가
            st.session_state["chat_messages"].append({
                "role": "assistant",
                "content": bot_response,
            })
        else:
            st.write("응답을 생성하지 못했습니다.")

Appending to frontend/app_chatbot.py


In [27]:
# [6.1 화면 띄우기] 브라우저에서 http://localhost:8501 을 엽니다
#   streamlit 을 PATH 에서 찾지 않고 커널의 파이썬으로 실행합니다 (sys.executable -m streamlit)
#   그래야 'streamlit: command not found' 가 나지 않습니다
import subprocess

if port_in_use(8501):
    print("8501번이 이미 떠 있습니다 -> http://localhost:8501")
else:
    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "frontend/app_chatbot.py",
         "--server.port", "8501", "--server.headless", "true"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(30):
        if port_in_use(8501):
            break
        time.sleep(1)
    print("프론트엔드: http://localhost:8501")
# 끄려면 : proc.terminate()

8501번이 이미 떠 있습니다 -> http://localhost:8501


In [28]:
# ==========================================================================
# [7. 통합 테스트]
# [7.1 인증] 키가 없거나 틀리면 401
# ==========================================================================
body = {"messages": [{"role": "user", "content": "안녕"}]}

r = requests.post(f"{API_BASE}/chat", json=body)
print(f"인증 없음 -> HTTP {r.status_code}")            # 401

r = requests.post(f"{API_BASE}/chat", json=body, headers={"X-API-Key": "wrong-key"})
print(f"잘못된 키 -> HTTP {r.status_code}")            # 401

r = requests.post(f"{API_BASE}/chat", json=body, headers={"X-API-Key": VALID_KEY}, timeout=120)
print(f"올바른 키 -> HTTP {r.status_code}")            # 200
print("응답:", r.json()["response"][:50])

인증 없음 -> HTTP 401
잘못된 키 -> HTTP 401


올바른 키 -> HTTP 200
응답: ームカミ ( 인스턴트 카드 ) : ROSE LOVER STRIPING EXCRD ULIT 


In [29]:
# [7.2 멀티턴 세 번] 대화 기록을 쌓아 가며 세 번 주고받습니다
messages = []
for user_msg in ["안녕하세요!", "오늘 뭐 하면 좋을까?", "맛있는 거 추천해줘"]:
    messages.append({"role": "user", "content": user_msg})
    r = requests.post(f"{API_BASE}/chat",
                      json={"messages": messages, "max_new_tokens": 50},
                      headers={"X-API-Key": VALID_KEY}, timeout=120)
    bot_msg = r.json()["response"]
    messages.append({"role": "bot", "content": bot_msg})   # 봇 응답도 기록에 추가
    print(f"사용자: {user_msg}\n봇    : {bot_msg[:60]}\n")

print("총 대화 턴:", len(messages) // 2)

사용자: 안녕하세요!
봇    : ! ( Mybugs. Baboard's boy, where create you his please a fee



사용자: 오늘 뭐 하면 좋을까?
봇    : 네가 왜 여기 왔지?



사용자: 맛있는 거 추천해줘
봇    : 너는 그냥 내가 먹는 것 뿐이야.

총 대화 턴: 3


In [30]:
# [7.3 입력 검증] 규격 밖이면 422
r = requests.post(f"{API_BASE}/chat", json={"messages": []},
                  headers={"X-API-Key": VALID_KEY})
print(f"빈 메시지         -> HTTP {r.status_code}")    # 422 (min_length=1)

r = requests.post(f"{API_BASE}/chat",
                  json={"messages": [{"role": "user", "content": "테스트"}], "temperature": 5.0},
                  headers={"X-API-Key": VALID_KEY})
print(f"temperature 초과  -> HTTP {r.status_code}")    # 422 (le=2.0)

빈 메시지         -> HTTP 422
temperature 초과  -> HTTP 422


In [31]:
# [7.4 동시 요청 4개]
#   서버의 추론 스레드는 2개(max_workers=2)라 4개를 보내면 2개씩 처리됩니다
#   뒤의 두 건이 더 오래 걸리는 것을 봅니다
#
# ==========================================================================
# [정리]
#   모델은 클래스 하나로 감싸고 서버는 그 클래스만 안다 -> ChatbotModel
#   모델은 시작할 때 한 번만 올린다                     -> @app.on_event("startup")
#   무거운 추론은 별도 스레드로                         -> run_in_executor
#   인증은 Depends 한 줄                                -> verify_api_key
#   입력 검증은 Pydantic                                -> Field(ge=, le=, min_length=) -> 422
#   서버는 대화를 기억하지 않는다                       -> 클라이언트가 messages 전체를 보냄
#   화면의 상태는 session_state                         -> st.session_state["chat_messages"]
#
#   답의 품질이 낮은 이유 : kogpt2는 대화용으로 학습되지 않은 작은 모델(1.25억 파라미터)입니다.
#   반복 억제 옵션으로 반복은 없앴지만, 내용을 좋게 하려면 대화 데이터로 파인튜닝하거나
#   대화용 모델로 바꿔야 합니다
# ==========================================================================
from concurrent.futures import ThreadPoolExecutor


def send_chat(i):
    start = time.time()
    r = requests.post(f"{API_BASE}/chat",
                      json={"messages": [{"role": "user", "content": f"질문 {i}번입니다"}],
                            "max_new_tokens": 30},
                      headers={"X-API-Key": VALID_KEY}, timeout=120)
    return {"id": i + 1, "elapsed": round(time.time() - start, 1), "status": r.status_code}


start = time.time()
with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(send_chat, range(4)))
for r in results:
    print(f"요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
print(f"전체: {round(time.time() - start, 1)}초")

요청 #1: 2.5초 (HTTP 200)
요청 #2: 2.4초 (HTTP 200)
요청 #3: 5.6초 (HTTP 200)
요청 #4: 5.6초 (HTTP 200)
전체: 5.6초
